# télos 3-Paradigm Throughput & Optimization Test Suite

This test suite benchmarks throughput (**steps/sec**, **tokens/sec**) and memory footprint across:
1. **Paradigms**: Autoregressive (AR), Masked Discrete Diffusion (MDLM), Uniform Noise Diffusion (UNDLM).
2. **Model Scales**: 5M, 12M, 25M, 50M, 100M parameters.
3. **Microbatch & Effective Batch Size Sweeps**: Microbatch 1 to 256, Effective Batch Size up to 4096 via Gradient Accumulation.

Goal: Discover optimal hardware saturation points, memory efficiency sweet spots, and compile/execution throughput limits on Apple Silicon Metal GPU.

In [ ]:
import os
import sys
import time
import gc
import math
import numpy as np
import pandas as pd
from pathlib import Path

# Ensure display function is available across both IPython and standard environments
try:
    from IPython.display import display
except ImportError:
    display = print

# Ensure working directory is project root
project_root = Path.cwd()
while not (project_root / "mdiff").exists() and project_root.parent != project_root:
    project_root = project_root.parent
os.chdir(project_root)
sys.path.insert(0, str(project_root))

import mlx.core as mx
import mlx.nn as mx_nn
import mlx.optimizers as mx_optim
from mlx.utils import tree_map

from mdiff.model.mlx_components import MLXTelosTransformer
from mdiff.training.trainer import apply_masking_mlx, loss_fn_mlx, build_special_token_lut, cast_optimizer_moments_bf16, get_sys_mem_str
from undiff.diffusion.forward_process import apply_uniform_noise_mlx
from undiff.diffusion.loss import undlm_loss
from ar.model.mlx_components import MLXCausalTransformer
from ar.training.trainer import ar_loss_fn_mlx

print("✓ MLX and all 3 paradigm components imported successfully!")

In [ ]:
# ── BENCHMARK RUNNER CORE ──

def benchmark_patch(
    paradigm: str,            # 'ar', 'mdlm', or 'undlm'
    model_cfg: dict,          # model arch dictionary
    micro_batch: int,         # microbatch size
    grad_accum: int,          # gradient accumulation steps
    warmup_steps: int = 3,    # graph trace warmup steps
    bench_steps: int = 10,     # benchmark measurement steps
    seq_len: int = 512,
    vocab_size: int = 8192
) -> dict:
    """Runs a short 10-20 step benchmark patch for a specific configuration."""
    mx.clear_cache()
    gc.collect()
    
    # 1. Instantiate Model
    if paradigm == "ar":
        model = MLXCausalTransformer(vocab_size=vocab_size, **model_cfg)
    else:
        model = MLXTelosTransformer(vocab_size=vocab_size, **model_cfg)
    model.set_dtype(mx.bfloat16)
    
    n_params = sum(p.size for _, p in mx_nn.utils.tree_flatten(model.parameters()))
    special_lut = build_special_token_lut(vocab_size)
    
    # 2. Paradigm-specific step function
    if paradigm == "ar":
        loss_and_grad = mx_nn.value_and_grad(model, ar_loss_fn_mlx)
        def microstep_raw(batch):
            (loss, ce), grads = loss_and_grad(model, batch, vocab_size)
            return loss, ce, grads
    elif paradigm == "mdlm":
        loss_and_grad = mx_nn.value_and_grad(model, loss_fn_mlx)
        def microstep_raw(batch):
            masked, mask_pos, t = apply_masking_mlx(batch, mask_token_id=1, special_token_lut=special_lut)
            (loss, ce), grads = loss_and_grad(model, masked, batch, mask_pos, t, vocab_size)
            return loss, ce, grads
    elif paradigm == "undlm":
        loss_and_grad = mx_nn.value_and_grad(model, undlm_loss)
        def microstep_raw(batch):
            noisy, corrupt_mask, t = apply_uniform_noise_mlx(batch, vocab_size=vocab_size, special_token_lut=special_lut)
            (loss, ce), grads = loss_and_grad(model, noisy, batch, t, vocab_size)
            return loss, ce, grads
    
    # Graph Trace Warmup
    dummy_seqs = mx.random.randint(0, vocab_size, (micro_batch, seq_len))
    dl, dc, dg = microstep_raw(dummy_seqs)
    mx.eval(dl, dc, dg)
    del dl, dc, dg
    
    state = [model.state]
    compiled_step = mx.compile(microstep_raw, inputs=state, outputs=state)
    optimizer = mx_optim.AdamW(learning_rate=3e-4, weight_decay=0.1)
    
    # 3. Warmup Execution
    for _ in range(warmup_steps):
        accum_grads = None
        for _ in range(grad_accum):
            batch = mx.random.randint(0, vocab_size, (micro_batch, seq_len))
            loss, ce, grads = compiled_step(batch)
            accum_grads = grads if accum_grads is None else tree_map(lambda a, b: a + b, accum_grads, grads)
            mx.eval(accum_grads, loss)
        accum_grads = tree_map(lambda g: g / grad_accum, accum_grads)
        optimizer.update(model, accum_grads)
        mx.eval(model.parameters(), optimizer.state)
    
    # Cast AdamW moments to bf16
    optimizer.state = cast_optimizer_moments_bf16(optimizer.state)
    mx.eval(optimizer.state)
    mx.clear_cache()
    
    # 4. Benchmark Measurement Loop
    start_time = time.perf_counter()
    for step in range(bench_steps):
        accum_grads = None
        for _ in range(grad_accum):
            batch = mx.random.randint(0, vocab_size, (micro_batch, seq_len))
            loss, ce, grads = compiled_step(batch)
            accum_grads = grads if accum_grads is None else tree_map(lambda a, b: a + b, accum_grads, grads)
            mx.eval(accum_grads, loss)
        accum_grads = tree_map(lambda g: g / grad_accum, accum_grads)
        optimizer.update(model, accum_grads)
        mx.eval(model.parameters(), optimizer.state)
    
    elapsed = time.perf_counter() - start_time
    
    steps_per_sec = bench_steps / elapsed
    eff_batch = micro_batch * grad_accum
    tok_per_sec = steps_per_sec * eff_batch * seq_len
    ms_per_step = (elapsed / bench_steps) * 1000.0
    peak_mem_gb = mx.get_peak_memory() / 1e9
    active_mem_gb = mx.get_active_memory() / 1e9
    
    del model, optimizer, compiled_step
    gc.collect()
    mx.clear_cache()
    
    return {
        "paradigm": paradigm.upper(),
        "params_m": round(n_params / 1e6, 2),
        "micro_batch": micro_batch,
        "grad_accum": grad_accum,
        "eff_batch": eff_batch,
        "steps_per_sec": round(steps_per_sec, 2),
        "tok_per_sec": int(tok_per_sec),
        "ms_per_step": round(ms_per_step, 1),
        "peak_mem_gb": round(peak_mem_gb, 2),
        "active_mem_gb": round(active_mem_gb, 2)
    }

In [ ]:
# ── TEST SUITE 1: MODEL SCALE SWEEP (5M to 100M) ACROSS ALL PARADIGMS ──
print("=" * 90)
print("TEST SUITE 1: MODEL SCALE SWEEP (5M -> 100M)")
print("=" * 90)

scales = {
    "5M":  {"d_model": 256, "n_layers": 4,  "n_heads": 4,  "n_kv_heads": 2},
    "12M": {"d_model": 256, "n_layers": 12, "n_heads": 8,  "n_kv_heads": 4},
    "25M": {"d_model": 512, "n_layers": 8,  "n_heads": 8,  "n_kv_heads": 4},
    "50M": {"d_model": 768, "n_layers": 8,  "n_heads": 12, "n_kv_heads": 4},
    "100M":{"d_model": 768, "n_layers": 16, "n_heads": 12, "n_kv_heads": 4}
}

results_scale = []
micro_b = 4
grad_a = 8

for scale_name, arch in scales.items():
    for paradigm in ["ar", "mdlm", "undlm"]:
        try:
            res = benchmark_patch(paradigm, arch, micro_batch=micro_b, grad_accum=grad_a)
            res["scale_tag"] = scale_name
            results_scale.append(res)
            print(f"  [{scale_name:<4} | {res['paradigm']:<5}] {res['steps_per_sec']:>5.2f} st/s | {res['tok_per_sec']:>8,} tok/s | {res['ms_per_step']:>6.1f} ms/st | Peak RAM: {res['peak_mem_gb']:.2f} GB")
        except Exception as e:
            print(f"  [{scale_name:<4} | {paradigm.upper():<5}] OOM / Error: {e}")

df_scale = pd.DataFrame(results_scale)
display(df_scale)

In [ ]:
# ── TEST SUITE 2: BATCH SIZE & ACCUMULATION SWEEP (10M / 12M SCALE) ──
print("=" * 90)
print("TEST SUITE 2: BATCH SIZE & GRAD ACCUM SWEEP (RESTRICTED TO ~10M MODEL)")
print("=" * 90)

arch_10m = {"d_model": 256, "n_layers": 12, "n_heads": 8, "n_kv_heads": 4}

# Sweep microbatch sizes from 1 up to 256
micro_batch_candidates = [1, 2, 4, 8, 16, 32, 64, 128, 256]
# Sweep effective batch sizes up to 4096
effective_batch_targets = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096]

results_batch = []

for mb in micro_batch_candidates:
    for eff in effective_batch_targets:
        if eff < mb or (eff % mb != 0):
            continue
        ga = eff // mb
        
        # Limit sweep density: test representative combinations
        if ga > 128 and mb > 16: continue
        
        for paradigm in ["ar", "mdlm", "undlm"]:
            try:
                res = benchmark_patch(paradigm, arch_10m, micro_batch=mb, grad_accum=ga, bench_steps=10)
                results_batch.append(res)
                print(f"  [{res['paradigm']:<5} | MB={mb:>3d} | GA={ga:>3d} | EffB={eff:>4d}] {res['steps_per_sec']:>5.2f} st/s | {res['tok_per_sec']:>9,} tok/s | Peak RAM: {res['peak_mem_gb']:.2f} GB")
            except Exception as e:
                print(f"  [{paradigm.upper():<5} | MB={mb:>3d} | GA={ga:>3d} | EffB={eff:>4d}] OOM / Error: {e}")
                break  # stop higher GA for this OOM microbatch

df_batch = pd.DataFrame(results_batch)
display(df_batch)

In [ ]:
# ── TEST SUITE SUMMARY & SWEET SPOT HIGHLIGHTS ──
print("=" * 90)
print("TOP THROUGHPUT CONFIGURATIONS PER PARADIGM")
print("=" * 90)

if not df_batch.empty:
    for p in ["AR", "MDLM", "UNDLM"]:
        sub = df_batch[df_batch["paradigm"] == p]
        if not sub.empty:
            best_tok = sub.loc[sub["tok_per_sec"].idxmax()]
            print(f"★ {p} Max Throughput: {best_tok['tok_per_sec']:,} tok/s ({best_tok['steps_per_sec']} st/s) at MicroBatch={best_tok['micro_batch']}, GradAccum={best_tok['grad_accum']} (EffB={best_tok['eff_batch']})")
            print(f"   Peak Memory: {best_tok['peak_mem_gb']} GB | Step Latency: {best_tok['ms_per_step']} ms\n")